# Sprint 4 — Data Inspection & Exploratory Data Analysis (EDA)

## Dataset documentation
- **Dataset:** Titanic passenger dataset from the Seaborn data repository.
- **Source:** https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv
- **Direct CSV:** https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv
- **Domain:** Historical passenger survival analysis.
- **Target variable:** `survived` (0 = did not survive, 1 = survived), if present.
- **Why selected:** It includes numeric and categorical columns, missing values, and a binary classification target.

The often-used version has approximately 891 rows and 15 columns. The code below prints the actual dimensions of the file loaded at runtime. Internet access is needed unless the CSV is downloaded locally and `DATA_URL` is changed.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(DATA_URL)
print("Loaded shape:", df.shape)
display(df.head())

**Learning approach:** For each concept, understand the explanation, connect it to a simple and business example, run the code, and explain what the output means. Treat findings as observations, not proof of causation.

# Notebook 3 — Data Quality Assessment
Identify and document issues. Do not fully preprocess the dataset in this sprint.

## Missing values and patterns

### 1. Concept
Missing values are absent observations. Counts and percentages quantify missingness; a pattern plot can reveal whether missing cells cluster.

### 2. Simple example
A form has unanswered questions.

### 3. Business example
Check which customer fields are incomplete and whether one team collects less information.

### 4. AI/ML relevance
Missing values may affect model reliability and later imputation choices.

In [ ]:
missing = pd.DataFrame({"count": df.isna().sum(), "percent": (df.isna().mean()*100).round(2)})
display(missing.sort_values("percent", ascending=False))
plt.figure(figsize=(11,4))
sns.heatmap(df.isna(), cbar=False, yticklabels=False)
plt.title("Missingness pattern")
plt.show()

### 5. Output explanation
The table reports missing counts and percentages; the heatmap shows where missing cells occur.

### 6. Interpretation / insight
Use the actual output to write a specific observation supported by a number, table, or plot. Note any uncertainty and do not claim causation.

### 7. Best practices
Check whether missingness is concentrated in a column or subgroup.

### 8. Common mistakes
Assuming missingness is random without evidence.

### 9. Advantages
Quantifies completeness.

### 10. Limitations
The pattern alone cannot explain why values are missing.

## Duplicate records

### 1. Concept
Fully duplicated rows match across all columns. Potential duplicates match on selected keys but may still be legitimate observations.

### 2. Simple example
Two copies of the same form differ from two separate forms for one person.

### 3. Business example
Find repeated orders before calculating totals.

### 4. AI/ML relevance
Duplicates can inflate group statistics or leak across train/test splits.

In [ ]:
print("Fully duplicated rows:", int(df.duplicated().sum()))
display(df[df.duplicated(keep=False)].head(20))
candidate = [c for c in ["name", "ticket"] if c in df.columns]
if candidate:
    print("Potential duplicates by", candidate, ":", int(df.duplicated(subset=candidate, keep=False).sum()))

### 5. Output explanation
The first count identifies exact repeated rows. Candidate-key duplicates require manual review.

### 6. Interpretation / insight
Use the actual output to write a specific observation supported by a number, table, or plot. Note any uncertainty and do not claim causation.

### 7. Best practices
Review the record definition before removing anything.

### 8. Common mistakes
Deleting all repeated names as if they were duplicate records.

### 9. Advantages
Detects repeated rows.

### 10. Limitations
Duplicate rules depend on the business meaning of a record.

## Invalid values, empty strings, and unexpected categories

### 1. Concept
Invalid values conflict with a field's meaning or accepted domain, such as a negative fare, empty text, or an unexpected category.

### 2. Simple example
A negative number of items or an unparseable date may be invalid.

### 3. Business example
Validate quantity, price, date, and status fields before reporting.

### 4. AI/ML relevance
Bad values can become misleading model signals.

In [ ]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    empty = int(df[col].astype("string").str.strip().eq("").fillna(False).sum())
    print(col, "empty/whitespace:", empty)
for col, rule in [("age", lambda s: s.lt(0) | s.gt(120)), ("fare", lambda s: s.lt(0))]:
    if col in df:
        print(col, "values outside illustrative range:", int(rule(df[col]).fillna(False).sum()))
for col in ["sex", "embarked"]:
    if col in df:
        print(col, df[col].value_counts(dropna=False).to_dict())

### 5. Output explanation
The code reports empty text, illustrative range violations, and observed category labels. Flagged values should be investigated, not automatically deleted.

### 6. Interpretation / insight
Use the actual output to write a specific observation supported by a number, table, or plot. Note any uncertainty and do not claim causation.

### 7. Best practices
Set validity rules based on column meaning and documentation.

### 8. Common mistakes
Assuming an unusual observation must be a mistake.

### 9. Advantages
Turns quality concerns into repeatable checks.

### 10. Limitations
Requires domain knowledge to define valid ranges.